In [10]:
import pandas as pd

In [11]:
df = pd.read_json("op.json",encoding="utf-16")

In [12]:
df.head(15)

,_meta,all_awardings,allow_live_comments,approved_at_utc,approved_by,archived,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,poll_data,preview,post_hint,link_flair_template_id,location_lat,location_long,location_name,websocket_url,num_duplicates,profile_img
0,"{'removal_type': 'deleted', 'retrieved_2nd_on'...",[],False,NaN,NaN,False,Perfect-Activity5471,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,{'retrieved_2nd_on': 1759386812},[],False,NaN,NaN,False,ichi3939,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,{'retrieved_2nd_on': 1759386938},[],False,NaN,NaN,False,ineedhope34,NaN,modmsg,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,{'retrieved_2nd_on': 1759387163},[],False,NaN,NaN,False,wellwellnow44,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"{'removal_type': 'automod_filtered', 'retrieve...",[],False,NaN,NaN,False,Intpshit,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,{'retrieved_2nd_on': 1759387249},[],False,NaN,NaN,False,Luke_tha_loop,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,{'retrieved_2nd_on': 1759387345},[],False,NaN,NaN,False,Dry_Remote263,NaN,modmsg,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,{'retrieved_2nd_on': 1759387573},[],False,NaN,NaN,False,Adorable_Egg7762,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,"{'removal_type': 'deleted', 'retrieved_2nd_on'...",[],False,NaN,NaN,False,Upstairs_Data_6430,NaN,NaN,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,{'retrieved_2nd_on': 1759387951},[],False,NaN,NaN,False,Izatron_,NaN,modmsg,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
counts = df.groupby('author').size()
counts[counts>2]
result = df[df['author'].isin(counts[counts > 2].index)]

In [14]:
result[['title','selftext']]

,title,selftext
0,I want a way out,If anyone has advice on this. Please let me kn...
2,I really was going to attempt today. Had prepp...,Same as above.\nI cannot talk about my true si...
3,There's no more hope for me,[removed]
4,"I already decided I will go- I need PRESENCE, ...","As you can read, I made my final decision.\n\n..."
9,One step and its over,.
...,...,...
163406,Still here,Parents saved me man i am alive because of par...
163407,Sometimes I like to imagine myself ripping apa...,"Not for ""there can only be one of me"" reasons,..."
163410,Struggling mentally scared.,I'm recovering from a really bad mental health...
163414,I got raped by my ex fiance and I cheated on h...,My fiance raped me in my sleep but his excuse ...


In [15]:
result.query("selftext != '[removed]'").index

Index([     0,      2,      4,      9,     18,     24,     25,     29,     32,
           37,
       ...
       163388, 163393, 163395, 163401, 163405, 163406, 163407, 163410, 163414,
       163415],
      dtype='int64', length=59304)

In [16]:
intermediary_df = result.query("selftext != '[removed]' and selftext.str.len() > 2")[['author','title','selftext']]

In [17]:
intermediary_df

,author,title,selftext
0,Perfect-Activity5471,I want a way out,If anyone has advice on this. Please let me kn...
2,ineedhope34,I really was going to attempt today. Had prepp...,Same as above.\nI cannot talk about my true si...
4,Intpshit,"I already decided I will go- I need PRESENCE, ...","As you can read, I made my final decision.\n\n..."
18,darkesthelp,Getting darker and darker,I try to hold on but the more I post on this s...
24,Other-Lab3485,Currently coming to the realization of just ho...,"I've never felt this alone before in my life,I..."
...,...,...,...
163406,Stunning_Doctor_2973,Still here,Parents saved me man i am alive because of par...
163407,ren_blackheart,Sometimes I like to imagine myself ripping apa...,"Not for ""there can only be one of me"" reasons,..."
163410,imscaredhelpme88,Struggling mentally scared.,I'm recovering from a really bad mental health...
163414,No_Recognition_121,I got raped by my ex fiance and I cheated on h...,My fiance raped me in my sleep but his excuse ...


In [18]:
import re
from tqdm import tqdm
tqdm.pandas()

def clean_and_spellcheck(text):
    if not isinstance(text, str):
        return []
    
    # Separate glued letters and numbers (e.g., '2929\n\n\nThat's' or 'mental14')
    text = re.sub(r'([a-zA-Z]+)(\d+)', r'\1 \2', text)
    
    # Normalize curly apostrophes to straight apostrophes
    text = text.replace("’", "'")
    
    # Extract clean alphabetic tokens
    clean_tokens = re.findall(r"[a-zA-Z]+(?:'[a-zA-Z]+)?", text.lower())
    
    return " ".join(clean_tokens)
intermediary_df['cleaned_selftext'] = intermediary_df['selftext'].progress_apply(clean_and_spellcheck)

100%|██████████| 58772/58772 [00:17<00:00, 3452.56it/s] 


In [19]:
import joblib
import torch
import torch.nn as nn
from tqdm import tqdm

# Load vectorizer
vectorizer = joblib.load("tfidf_vectorizer.pkl")

input_size = len(vectorizer.get_feature_names_out())

tqdm.pandas()

model = nn.Sequential(
    nn.Linear(input_size, 16),
    nn.ReLU(),
    nn.Linear(16, 10),
    nn.ReLU(),
    nn.Linear(10, 1)
)

# Load trained weights
model.load_state_dict(
    torch.load(
        "suicide_classifier.pth",
        map_location="cpu",
        weights_only=True
    )
)

# Load model
model.load_state_dict(torch.load("suicide_classifier.pth", weights_only=True))
model.eval()

# new_text = ["""
# i just wanna give up. I'm tired.

# i don't have money but I guess I could find a way to get a bottle of silver cleaner. i just want everything to end.
# """]

def pred(new_text:str):
    X_new = vectorizer.transform([new_text])

    X_new_tensor = torch.tensor(X_new.toarray(), dtype=torch.float32)

    with torch.no_grad():
        logits = model(X_new_tensor)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

    # print("Probability:", probabilities.item())
    return("Prediction:", predictions.item())


intermediary_df['suicidal_text'] = intermediary_df['cleaned_selftext'].progress_apply(pred)

100%|██████████| 58772/58772 [00:42<00:00, 1371.21it/s]


In [20]:
intermediary_df.groupby('author').size()

author
--Bunny_Boy--        2
--Tyler_durden-      2
--jyushimatsudesu    2
--piplup--           2
--turbulence--       5
                    ..
zombiemaxxer         4
zombit3z             4
zumodecarne          2
zy-zee               2
zzwolfy              4
Length: 11510, dtype: int64

In [21]:
list(df.columns)

['_meta',
 'all_awardings',
 'allow_live_comments',
 'approved_at_utc',
 'approved_by',
 'archived',
 'author',
 'author_flair_background_color',
 'author_flair_css_class',
 'author_flair_richtext',
 'author_flair_template_id',
 'author_flair_text',
 'author_flair_text_color',
 'author_flair_type',
 'author_fullname',
 'author_is_blocked',
 'author_patreon_flair',
 'author_premium',
 'awarders',
 'banned_at_utc',
 'banned_by',
 'can_gild',
 'can_mod_post',
 'category',
 'clicked',
 'content_categories',
 'contest_mode',
 'created',
 'created_utc',
 'discussion_type',
 'distinguished',
 'domain',
 'downs',
 'edited',
 'gilded',
 'gildings',
 'hidden',
 'hide_score',
 'id',
 'is_created_from_ads_ui',
 'is_crosspostable',
 'is_meta',
 'is_original_content',
 'is_reddit_media_domain',
 'is_robot_indexable',
 'is_self',
 'is_video',
 'likes',
 'link_flair_background_color',
 'link_flair_css_class',
 'link_flair_richtext',
 'link_flair_text',
 'link_flair_text_color',
 'link_flair_type',
 'l

In [ ]:
user_df = pd.read_json('u_yaoimalover101_posts.jsonl',lines=True)

In [ ]:
user_df[['subreddit','title','selftext','created_utc']]

In [ ]:
user_df[['subreddit','title','selftext','created_utc']].query("subreddit == 'SuicideWatch'")

In [ ]:
filtered_df = user_df[['subreddit', 'title', 'selftext', 'created_utc']].copy()
filtered_df['created_ist'] = (
    pd.to_datetime(filtered_df['created_utc'], unit='s', utc=True)
    .dt.tz_convert('Asia/Kolkata')
)

In [ ]:
filtered_df